# arg-position-back-functions — worked example 2: Write atan2_back0 and atan2_back1 for out = atan2(y, x)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `arg-position-back-functions`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The per-argument-position convention shines on a genuinely asymmetric op. For `out = atan2(a, b)` (angle of the point `(b, a)`), the gradient w.r.t. arg 0 (`a`) and arg 1 (`b`) are different rational functions of `a` and `b`. We still write one back fn per position, each with signature `(grad_out, out, a, b)`.

## Worked solution

**Goal.** Implement back fns for `out = atan2(a, b)`, where arg 0 is `a` and arg 1 is `b`.

**Step 1 — recall the derivatives.** With `r2 = a**2 + b**2`, the standard results are `d atan2(a,b)/da = b / r2` and `d atan2(a,b)/db = -a / r2`. (Think of `atan2` as the angle; moving `a` and `b` rotates the point in opposite rotational senses, hence the sign difference.)

**Step 2 — compute the shared denominator once.** Both back fns need `r2 = a*a + b*b`. Computing it inside each fn keeps them self-contained; it is identical work either way.

**Step 3 — apply the chain rule.** `dL/da = grad_out * (b / r2)` and `dL/db = grad_out * (-a / r2)`. So `atan2_back0` multiplies the upstream grad by `b / r2`, while `atan2_back1` multiplies by `-a / r2`.

**Step 4 — verify against autograd.** The asymmetry (numerator `b` vs `-a`) is exactly why you cannot reuse one back fn for both positions. We cross-check both against `torch.atan2`'s autograd grads.

In [ ]:
def atan2_back0(grad_out: Tensor, out: Tensor, a: Tensor, b: Tensor) -> Tensor:
    # d atan2(a, b)/da = b / (a^2 + b^2)
    r2 = a * a + b * b
    return grad_out * (b / r2)


def atan2_back1(grad_out: Tensor, out: Tensor, a: Tensor, b: Tensor) -> Tensor:
    # d atan2(a, b)/db = -a / (a^2 + b^2)
    r2 = a * a + b * b
    return grad_out * (-a / r2)


t.manual_seed(0)
a = t.randn(4, requires_grad=True)
b = t.randn(4, requires_grad=True)
out = t.atan2(a, b)
grad_out = t.randn(4)
out.backward(grad_out)

ga = atan2_back0(grad_out, out.detach(), a.detach(), b.detach())
gb = atan2_back1(grad_out, out.detach(), a.detach(), b.detach())
print('grad_a match:', t.allclose(ga, a.grad, atol=1e-6))
print('grad_b match:', t.allclose(gb, b.grad, atol=1e-6))